# Backtest Framework Comparsion

This notebook compares the same 50/200 SMA crossover portfolio across data vendors and backtesting frameworks. The point is not to make SMA crossover a platform abstraction; it is a compact test case for whether Quant Orchestrator can hold the strategy constant while changing data providers and execution engines.

Data and SMA features come from Quant Warehouse. The backtesting engines receive prepared in-memory frames and do not compute their own indicators. The notebook also controls two common sources of false vendor differences: every symbol is aligned to the common provider date intersection, and Yahoo OHLC prices are dividend-adjusted from Quant Warehouse dividend fields so they are on the same adjusted-price basis as FMP history.

In [1]:
from __future__ import annotations

from time import perf_counter
import warnings

warnings.filterwarnings("ignore", message="Jupyter Notebook detected.*", category=UserWarning)
warnings.filterwarnings("ignore", message="DataFrameGroupBy.apply operated on the grouping columns.*", category=FutureWarning)
warnings.filterwarnings("ignore", message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*", category=FutureWarning)

import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse import Warehouse
from quant_orchestrator.backtests.research import (
    FrameworkRun,
    _coerce_framework_run,
    _framework_comparison_rows,
    _run_nautilus_isolated,
    build_sma_frame,
    run_backtesting_py,
    run_zipline,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    OHLCV_COLUMNS,
    combine_equity_curves,
    equal_notional_capital,
)
from quant_orchestrator.strategy import summarize_backtest

## Setup

The experiment is a single MAG7 portfolio, not seven independent headline backtests. Each symbol gets an equal capital sleeve and the portfolio equity curve is the sum of those sleeves. We then compare the portfolio-level result for each provider/framework pair.

To keep the comparison fair, all providers use the same global symbol/provider calendar after loading, and all OHLC inputs are on an adjusted-price basis before Quant Warehouse computes SMA features.

In [2]:
SYMBOLS = MAG7_SYMBOLS
PROVIDERS = ("yfinance", "fmp")
FRAMEWORKS = ("backtesting.py", "zipline", "nautilus")
START = "2020-01-01"
END = None
FAST_WINDOW = 50
SLOW_WINDOW = 200
CAPITAL_BASE = 100_000.0
PRICE_BASIS = "dividend_adjusted_ohlc"
DATE_ALIGNMENT = "global_provider_symbol_intersection"

CONFIG = pd.DataFrame(
    [
        {
            "symbols": ", ".join(SYMBOLS),
            "providers": ", ".join(PROVIDERS),
            "frameworks": ", ".join(FRAMEWORKS),
            "start": START,
            "end": END or "latest available",
            "fast_window": FAST_WINDOW,
            "slow_window": SLOW_WINDOW,
            "capital_base": CAPITAL_BASE,
            "capital_per_symbol": equal_notional_capital(CAPITAL_BASE, len(SYMBOLS)),
            "price_basis": PRICE_BASIS,
            "date_alignment": DATE_ALIGNMENT,
        }
    ]
)
display(CONFIG.T.rename(columns={0: "value"}))

,value
symbols,"AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA"
providers,"yfinance, fmp"
frameworks,"backtesting.py, zipline, nautilus"
start,2020-01-01
end,latest available
fast_window,50
slow_window,200
capital_base,100000.0
capital_per_symbol,14285.714286
price_basis,dividend_adjusted_ohlc


In [3]:
def _normalize_index(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.rename(columns=str.lower).copy()
    normalized.index = pd.DatetimeIndex(normalized.index)
    if normalized.index.tz is not None:
        normalized.index = normalized.index.tz_convert(None)
    return normalized.sort_index()


def _yfinance_dividend_adjustment_factor(frame: pd.DataFrame) -> pd.Series:
    dividends = frame.get("yfinance__dividend", pd.Series(0.0, index=frame.index)).fillna(0.0)
    close = frame["close"].astype(float)
    factor = pd.Series(1.0, index=frame.index, dtype=float)

    for index_position in range(len(frame) - 2, -1, -1):
        next_position = index_position + 1
        dividend = float(dividends.iloc[next_position])
        if dividend:
            previous_close = float(close.iloc[index_position])
            factor.iloc[index_position] = factor.iloc[next_position] * max((previous_close - dividend) / previous_close, 0.0)
        else:
            factor.iloc[index_position] = factor.iloc[next_position]
    return factor


def standardize_price_basis(frame: pd.DataFrame, *, provider: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    standardized = _normalize_index(frame)
    raw_close = standardized["close"].astype(float).copy()

    if provider == "yfinance":
        adjustment_factor = _yfinance_dividend_adjustment_factor(standardized)
        for column in ("open", "high", "low", "close"):
            standardized[column] = standardized[column].astype(float) * adjustment_factor
        basis_method = "yfinance_dividend_adjusted_from_warehouse_dividends"
    else:
        adjustment_factor = pd.Series(1.0, index=standardized.index, dtype=float)
        basis_method = "provider_adjusted_history_as_stored"

    standardized = standardized.loc[:, list(OHLCV_COLUMNS)].copy()
    standardized["volume"] = standardized["volume"].fillna(0.0)
    audit = pd.DataFrame(
        [
            {
                "provider": provider,
                "raw_start_close": float(raw_close.iloc[0]),
                "standardized_start_close": float(standardized["close"].iloc[0]),
                "raw_end_close": float(raw_close.iloc[-1]),
                "standardized_end_close": float(standardized["close"].iloc[-1]),
                "min_adjustment_factor": float(adjustment_factor.min()),
                "max_adjustment_factor": float(adjustment_factor.max()),
                "basis_method": basis_method,
            }
        ]
    )
    return standardized, audit


def load_all_provider_prices() -> dict[str, dict[str, pd.DataFrame]]:
    warehouse = Warehouse()
    return {
        provider: {
            symbol: warehouse.read_prices(symbol, provider=provider, start=START, end=END)
            for symbol in SYMBOLS
        }
        for provider in PROVIDERS
    }


def prepare_comparison_frames(
    raw_frames: dict[str, dict[str, pd.DataFrame]],
) -> tuple[dict[str, dict[str, pd.DataFrame]], pd.DataFrame, pd.DataFrame]:
    standardized: dict[str, dict[str, pd.DataFrame]] = {provider: {} for provider in PROVIDERS}
    basis_audits = []

    for provider in PROVIDERS:
        for symbol in SYMBOLS:
            frame, audit = standardize_price_basis(raw_frames[provider][symbol], provider=provider)
            standardized[provider][symbol] = frame
            audit.insert(0, "symbol", symbol)
            basis_audits.append(audit)

    global_index = None
    for symbol in SYMBOLS:
        for provider in PROVIDERS:
            provider_index = standardized[provider][symbol].index
            global_index = provider_index if global_index is None else global_index.intersection(provider_index)
    global_index = pd.DatetimeIndex(global_index).sort_values()
    if global_index.empty:
        raise ValueError("No common dates across all symbols and providers")

    aligned: dict[str, dict[str, pd.DataFrame]] = {provider: {} for provider in PROVIDERS}
    date_rows = []
    for symbol in SYMBOLS:
        for provider in PROVIDERS:
            before = standardized[provider][symbol]
            after = before.loc[global_index].copy()
            aligned[provider][symbol] = after
            date_rows.append(
                {
                    "symbol": symbol,
                    "provider": provider,
                    "raw_start": before.index.min().date().isoformat(),
                    "raw_end": before.index.max().date().isoformat(),
                    "raw_rows": len(before),
                    "aligned_start": after.index.min().date().isoformat(),
                    "aligned_end": after.index.max().date().isoformat(),
                    "aligned_rows": len(after),
                    "dropped_rows": len(before) - len(after),
                }
            )

    basis_audit = pd.concat(basis_audits, ignore_index=True)
    first_close = (
        pd.concat(
            [
                aligned[provider][symbol]["close"].rename((symbol, provider))
                for symbol in SYMBOLS
                for provider in PROVIDERS
            ],
            axis=1,
        )
        .iloc[0]
        .rename("first_aligned_standardized_close")
        .reset_index()
        .rename(columns={"level_0": "symbol", "level_1": "provider"})
    )
    basis_audit = basis_audit.merge(first_close, on=["symbol", "provider"], how="left")
    return aligned, pd.DataFrame(date_rows), basis_audit

def run_symbol_sleeve(
    framework: str,
    *,
    symbol: str,
    prices: pd.DataFrame,
    capital_base: float,
) -> FrameworkRun:
    if framework == "nautilus":
        return _run_nautilus_isolated(
            prices,
            symbol=symbol,
            fast_window=FAST_WINDOW,
            slow_window=SLOW_WINDOW,
            capital_base=capital_base,
        )

    signal_frame = build_sma_frame(prices, fast_window=FAST_WINDOW, slow_window=SLOW_WINDOW)
    if framework == "backtesting.py":
        return _coerce_framework_run(run_backtesting_py(signal_frame, symbol=symbol, capital_base=capital_base))
    if framework == "zipline":
        return _coerce_framework_run(run_zipline(signal_frame, symbol=symbol, capital_base=capital_base))
    raise ValueError(f"Unsupported framework: {framework}")


def run_mag7_portfolio(
    *,
    provider: str,
    framework: str,
    price_frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, FrameworkRun]]:
    capital_per_symbol = equal_notional_capital(CAPITAL_BASE, len(price_frames))
    started = perf_counter()
    sleeves = []
    runs = {}
    symbol_rows = []
    trades = 0
    native_elapsed_seconds = 0.0

    for symbol, prices in price_frames.items():
        run = run_symbol_sleeve(
            framework,
            symbol=symbol,
            prices=prices,
            capital_base=capital_per_symbol,
        )
        runs[symbol] = run
        sleeves.append(run.equity.rename(symbol))
        row = run.summary.copy()
        row["provider"] = provider
        row["framework"] = framework
        symbol_rows.append(row)
        trades += int(run.summary["trades"].iloc[0])
        native_elapsed_seconds += float(run.summary["elapsed_seconds"].iloc[0])

    portfolio_equity = combine_equity_curves(sleeves)
    wall_clock_seconds = perf_counter() - started
    portfolio_summary = summarize_backtest(
        framework=framework,
        symbol="MAG7",
        equity=portfolio_equity,
        elapsed_seconds=wall_clock_seconds,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary["provider"] = provider
    portfolio_summary["fast_window"] = FAST_WINDOW
    portfolio_summary["slow_window"] = SLOW_WINDOW
    portfolio_summary["capital_base"] = CAPITAL_BASE
    portfolio_summary["capital_per_symbol"] = capital_per_symbol
    portfolio_summary["price_basis"] = PRICE_BASIS
    portfolio_summary["date_alignment"] = DATE_ALIGNMENT
    portfolio_summary["native_elapsed_seconds"] = round(native_elapsed_seconds, 4)
    portfolio_summary["wall_clock_seconds"] = round(wall_clock_seconds, 4)
    portfolio_summary["performance_score"] = (
        portfolio_summary["total_return"] + portfolio_summary["max_drawdown"]
    )
    return portfolio_summary, pd.concat(symbol_rows, ignore_index=True), runs

## Run Jobs

Each row below is a Quant Orchestrator-style job: one data provider, one backtesting framework, one equal-notional MAG7 portfolio strategy. The first two outputs are audit tables showing the price-basis standardization and common-date alignment applied before any backtest runs.

In [4]:
raw_price_frames = load_all_provider_prices()
price_frames_by_provider, date_alignment_audit, price_basis_audit = prepare_comparison_frames(raw_price_frames)

display(date_alignment_audit)
display(price_basis_audit)

portfolio_rows = []
symbol_rows = []
runs: dict[str, dict[str, dict[str, FrameworkRun]]] = {}

for provider in PROVIDERS:
    runs[provider] = {}
    for framework in FRAMEWORKS:
        portfolio_summary, symbol_summary, framework_runs = run_mag7_portfolio(
            provider=provider,
            framework=framework,
            price_frames=price_frames_by_provider[provider],
        )
        portfolio_rows.append(portfolio_summary)
        symbol_rows.append(symbol_summary)
        runs[provider][framework] = framework_runs

comparison = (
    pd.concat(portfolio_rows, ignore_index=True)
    .sort_values(["provider", "framework"])
    .reset_index(drop=True)
)
symbol_detail = pd.concat(symbol_rows, ignore_index=True).sort_values(["provider", "framework", "symbol"]).reset_index(drop=True)
comparison

,symbol,provider,raw_start,raw_end,raw_rows,aligned_start,aligned_end,aligned_rows,dropped_rows
0,AAPL,yfinance,2020-01-02,2026-06-23,1626,2023-01-03,2026-06-23,870,756
1,AAPL,fmp,2020-01-02,2026-06-24,1627,2023-01-03,2026-06-23,870,757
2,AMZN,yfinance,2023-01-03,2026-06-23,870,2023-01-03,2026-06-23,870,0
3,AMZN,fmp,2020-01-02,2026-06-24,1627,2023-01-03,2026-06-23,870,757
4,GOOGL,yfinance,2023-01-03,2026-06-23,870,2023-01-03,2026-06-23,870,0
5,GOOGL,fmp,2020-01-02,2026-06-24,1627,2023-01-03,2026-06-23,870,757
6,META,yfinance,2023-01-03,2026-06-23,870,2023-01-03,2026-06-23,870,0
7,META,fmp,2020-01-02,2026-06-24,1627,2023-01-03,2026-06-23,870,757
8,MSFT,yfinance,2023-01-03,2026-06-23,870,2023-01-03,2026-06-23,870,0
9,MSFT,fmp,2020-01-02,2026-06-24,1627,2023-01-03,2026-06-23,870,757


,symbol,provider,raw_start_close,standardized_start_close,raw_end_close,standardized_end_close,min_adjustment_factor,max_adjustment_factor,basis_method,first_aligned_standardized_close
0,AAPL,yfinance,75.087502,72.333873,294.299988,294.299988,0.963328,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,122.982715
1,AMZN,yfinance,85.820000,85.820000,234.110001,234.110001,1.000000,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,85.820000
2,GOOGL,yfinance,89.120003,88.336700,346.130005,346.130005,0.991211,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,88.336700
3,META,yfinance,124.739998,123.654126,562.200012,562.200012,0.991295,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,123.654126
4,MSFT,yfinance,239.580002,232.948268,373.940002,373.940002,0.972319,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,232.948268
5,NVDA,yfinance,14.315000,14.283262,200.039993,200.039993,0.997783,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,14.283262
6,TSLA,yfinance,108.099998,108.099998,381.609985,381.609985,1.000000,1.0,yfinance_dividend_adjusted_from_warehouse_divi...,108.099998
7,AAPL,fmp,72.330000,72.330000,293.080000,293.080000,1.000000,1.0,provider_adjusted_history_as_stored,122.970000
8,AMZN,fmp,94.900000,94.900000,234.270000,234.270000,1.000000,1.0,provider_adjusted_history_as_stored,85.820000
9,GOOGL,fmp,68.430000,68.430000,345.290000,345.290000,1.000000,1.0,provider_adjusted_history_as_stored,89.120000


Loading BokehJS ...

,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,...,provider,fast_window,slow_window,capital_base,capital_per_symbol,price_basis,date_alignment,native_elapsed_seconds,wall_clock_seconds,performance_score
0,backtesting.py,MAG7,2023-01-03,2026-06-23,870,19,129206.83,0.2921,-0.1639,0.0078,...,fmp,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.1325,0.2202,0.1282
1,nautilus,MAG7,2023-01-03,2026-06-23,870,35,140474.19,0.4047,-0.1623,0.0094,...,fmp,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.2061,20.5426,0.2424
2,zipline,MAG7,2023-01-03,2026-06-23,870,40,140956.85,0.4096,-0.1641,0.0092,...,fmp,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,1.1874,1.5518,0.2455
3,backtesting.py,MAG7,2023-01-03,2026-06-23,870,18,129536.93,0.2954,-0.1631,0.0078,...,yfinance,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.1336,0.4901,0.1323
4,nautilus,MAG7,2023-01-03,2026-06-23,870,33,141221.00,0.4122,-0.1615,0.0094,...,yfinance,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.2048,20.1450,0.2507
5,zipline,MAG7,2023-01-03,2026-06-23,870,38,141791.99,0.4179,-0.1641,0.0092,...,yfinance,50,200,100000.0,14285.714286,dividend_adjusted_ohlc,global_provider_symbol_intersection,1.3513,2.0063,0.2538


## Portfolio Comparison

This table is the main result. Returns and drawdowns are portfolio-level values after combining the equal-notional symbol sleeves. The `start` and `end` columns should now match by framework across providers because every symbol/provider frame was clipped to the same global intersection before running.

In [5]:
comparison_display = comparison.loc[:, [
    "provider",
    "framework",
    "symbol",
    "start",
    "end",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "daily_vol",
    "performance_score",
    "price_basis",
    "date_alignment",
    "wall_clock_seconds",
    "bars_per_second",
]].copy()
for column in ["total_return", "max_drawdown", "daily_vol", "performance_score"]:
    comparison_display[column] = (comparison_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "wall_clock_seconds", "bars_per_second"]:
    comparison_display[column] = comparison_display[column].round(2)
display(comparison_display)

,provider,framework,symbol,start,end,trades,final_equity,total_return,max_drawdown,daily_vol,performance_score,price_basis,date_alignment,wall_clock_seconds,bars_per_second
0,fmp,backtesting.py,MAG7,2023-01-03,2026-06-23,19,129206.83,29.21%,-16.39%,0.78%,12.82%,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.22,3950.57
1,fmp,nautilus,MAG7,2023-01-03,2026-06-23,35,140474.19,40.47%,-16.23%,0.94%,24.24%,dividend_adjusted_ohlc,global_provider_symbol_intersection,20.54,42.35
2,fmp,zipline,MAG7,2023-01-03,2026-06-23,40,140956.85,40.96%,-16.41%,0.92%,24.55%,dividend_adjusted_ohlc,global_provider_symbol_intersection,1.55,560.63
3,yfinance,backtesting.py,MAG7,2023-01-03,2026-06-23,18,129536.93,29.54%,-16.31%,0.78%,13.23%,dividend_adjusted_ohlc,global_provider_symbol_intersection,0.49,1775.19
4,yfinance,nautilus,MAG7,2023-01-03,2026-06-23,33,141221.00,41.22%,-16.15%,0.94%,25.07%,dividend_adjusted_ohlc,global_provider_symbol_intersection,20.14,43.19
5,yfinance,zipline,MAG7,2023-01-03,2026-06-23,38,141791.99,41.79%,-16.41%,0.92%,25.38%,dividend_adjusted_ohlc,global_provider_symbol_intersection,2.01,433.64


## Symbol Detail

The strategy is reported as a MAG7 portfolio, but the symbol-level sleeves are shown here to make the portfolio construction auditable.

In [6]:
symbol_detail_display = symbol_detail.loc[:, [
    "provider",
    "framework",
    "symbol",
    "trades",
    "final_equity",
    "total_return",
    "max_drawdown",
    "elapsed_seconds",
]].copy()
for column in ["total_return", "max_drawdown"]:
    symbol_detail_display[column] = (symbol_detail_display[column] * 100).round(2).astype(str) + "%"
for column in ["final_equity", "elapsed_seconds"]:
    symbol_detail_display[column] = symbol_detail_display[column].round(2)
display(symbol_detail_display)

,provider,framework,symbol,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,backtesting.py,AAPL,3,15166.15,6.16%,-14.41%,0.02
1,fmp,backtesting.py,AMZN,3,14274.85,-0.08%,-24.3%,0.02
2,fmp,backtesting.py,GOOGL,2,20470.91,43.3%,-15.25%,0.02
3,fmp,backtesting.py,META,2,20215.27,41.51%,-27.96%,0.02
4,fmp,backtesting.py,MSFT,4,14293.83,0.06%,-11.97%,0.02
5,fmp,backtesting.py,NVDA,1,32744.08,129.21%,-25.89%,0.02
6,fmp,backtesting.py,TSLA,4,12041.71,-15.71%,-43.42%,0.02
7,fmp,nautilus,AAPL,5,14982.29,4.88%,-14.36%,0.03
8,fmp,nautilus,AMZN,5,14181.57,-0.73%,-25.48%,0.03
9,fmp,nautilus,GOOGL,3,20713.71,45.0%,-15.21%,0.03


## Difference Attribution

This is a small two-way decomposition over the 2 providers x 3 frameworks matrix. It is not a statistical test; it is a practical magnitude check for whether the provider choice or framework choice moved a metric more in this fixed strategy example.

In [7]:
factor_report = pd.concat(
    [
        _framework_comparison_rows(comparison, metric="performance_score"),
        _framework_comparison_rows(comparison, metric="total_return"),
        _framework_comparison_rows(comparison, metric="max_drawdown"),
        _framework_comparison_rows(comparison, metric="wall_clock_seconds"),
    ],
    ignore_index=True,
)
factor_display = factor_report.copy()
for column in ["framework_share", "provider_share", "residual_share"]:
    factor_display[column] = (factor_display[column] * 100).round(2).astype(str) + "%"
for column in ["framework_ss", "provider_ss", "residual_ss", "provider_to_framework_ratio"]:
    factor_display[column] = factor_display[column].round(6)
display(factor_display)

,metric,framework_ss,provider_ss,residual_ss,framework_share,provider_share,residual_share,dominant_factor,provider_to_framework_ratio
0,performance_score,0.018528,0.000071,0.000006,99.58%,0.38%,0.03%,framework,0.003854
1,total_return,0.018389,0.000061,0.000007,99.63%,0.33%,0.04%,framework,0.003306
2,max_drawdown,0.000005,0.000000,0.000000,88.99%,7.34%,3.67%,framework,0.082474
3,wall_clock_seconds,497.482375,0.017800,0.200951,99.96%,0.0%,0.04%,framework,0.000036


## Summaries

These summaries average over the other axis. They are useful for quick scanning, but the detailed comparison table above is the source of truth for this run.

In [8]:
framework_summary = comparison.groupby("framework").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)
provider_summary = comparison.groupby("provider").agg(
    mean_total_return=("total_return", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_performance_score=("performance_score", "mean"),
    mean_wall_clock_seconds=("wall_clock_seconds", "mean"),
    mean_bars_per_second=("bars_per_second", "mean"),
).sort_values("mean_performance_score", ascending=False)

display(framework_summary)
display(provider_summary)

,mean_total_return,mean_max_drawdown,mean_performance_score,mean_wall_clock_seconds,mean_bars_per_second
framework,,,,,
zipline,0.41375,-0.1641,0.24965,1.77905,497.135
nautilus,0.40845,-0.1619,0.24655,20.34380,42.770
backtesting.py,0.29375,-0.1635,0.13025,0.35515,2862.880


,mean_total_return,mean_max_drawdown,mean_performance_score,mean_wall_clock_seconds,mean_bars_per_second
provider,,,,,
yfinance,0.375167,-0.162900,0.212267,7.547133,750.673333
fmp,0.368800,-0.163433,0.205367,7.438200,1517.850000


## Written Analysis

The following cell writes the analysis from the outputs above so it changes when the notebook is rerun with a different date range, provider set, framework set, or symbol universe.

In [9]:
def _fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"


def _metric_factor(metric: str) -> pd.Series:
    return factor_report.loc[factor_report["metric"] == metric].iloc[0]


best_score = comparison.sort_values("performance_score", ascending=False).iloc[0]
best_return = comparison.sort_values("total_return", ascending=False).iloc[0]
fastest = comparison.sort_values("wall_clock_seconds", ascending=True).iloc[0]
score_factor = _metric_factor("performance_score")
return_factor = _metric_factor("total_return")
speed_factor = _metric_factor("wall_clock_seconds")
max_dropped_rows = int(date_alignment_audit["dropped_rows"].max())
yahoo_adjustment = price_basis_audit.loc[price_basis_audit["provider"] == "yfinance", "min_adjustment_factor"].min()

analysis = f"""## Analysis From This Run

- Best risk-adjusted simple score: **{best_score['provider']} + {best_score['framework']}** with performance score {_fmt_pct(float(best_score['performance_score']))}, total return {_fmt_pct(float(best_score['total_return']))}, and max drawdown {_fmt_pct(float(best_score['max_drawdown']))}.
- Best raw return: **{best_return['provider']} + {best_return['framework']}** with total return {_fmt_pct(float(best_return['total_return']))}.
- Fastest run: **{fastest['provider']} + {fastest['framework']}** at {float(fastest['wall_clock_seconds']):.2f} wall-clock seconds for the full MAG7 portfolio run.
- Date coverage is controlled with `{DATE_ALIGNMENT}`. The largest provider-specific row drop for any symbol was {max_dropped_rows} rows before backtesting.
- Price basis is controlled with `{PRICE_BASIS}`. Yahoo OHLC was dividend-adjusted from Quant Warehouse dividend fields; the smallest Yahoo adjustment factor in this universe was {yahoo_adjustment:.4f}. FMP was treated as adjusted history as stored.
- For performance score, the larger observed driver was **{score_factor['dominant_factor']}**: framework share {score_factor['framework_share']:.1%}, provider share {score_factor['provider_share']:.1%}, residual share {score_factor['residual_share']:.1%}.
- For raw total return, the larger observed driver was **{return_factor['dominant_factor']}**: framework share {return_factor['framework_share']:.1%}, provider share {return_factor['provider_share']:.1%}, residual share {return_factor['residual_share']:.1%}.
- For speed, the larger observed driver was **{speed_factor['dominant_factor']}**: framework share {speed_factor['framework_share']:.1%}, provider share {speed_factor['provider_share']:.1%}, residual share {speed_factor['residual_share']:.1%}.

Interpretation: after controlling date coverage and adjusted-price basis, remaining vendor differences are more likely to come from vendor OHLC/volume quality, rounding, missing bars, or corporate-action handling inside the vendor history rather than obvious adjusted-vs-unadjusted data or unequal date windows.
"""
display(Markdown(analysis))

## Analysis From This Run

- Best risk-adjusted simple score: **yfinance + zipline** with performance score 25.38%, total return 41.79%, and max drawdown -16.41%.
- Best raw return: **yfinance + zipline** with total return 41.79%.
- Fastest run: **fmp + backtesting.py** at 0.22 wall-clock seconds for the full MAG7 portfolio run.
- Date coverage is controlled with `global_provider_symbol_intersection`. The largest provider-specific row drop for any symbol was 757 rows before backtesting.
- Price basis is controlled with `dividend_adjusted_ohlc`. Yahoo OHLC was dividend-adjusted from Quant Warehouse dividend fields; the smallest Yahoo adjustment factor in this universe was 0.9633. FMP was treated as adjusted history as stored.
- For performance score, the larger observed driver was **framework**: framework share 99.6%, provider share 0.4%, residual share 0.0%.
- For raw total return, the larger observed driver was **framework**: framework share 99.6%, provider share 0.3%, residual share 0.0%.
- For speed, the larger observed driver was **framework**: framework share 100.0%, provider share 0.0%, residual share 0.0%.

Interpretation: after controlling date coverage and adjusted-price basis, remaining vendor differences are more likely to come from vendor OHLC/volume quality, rounding, missing bars, or corporate-action handling inside the vendor history rather than obvious adjusted-vs-unadjusted data or unequal date windows.
